# EML / Tamari exploration notebook

This notebook reproduces the Log‑Exp slice visualization, enumerates small EML trees (binary trees), computes Loday coordinates, constructs the Tamari graph for a fixed size, computes the discrete cost Φ (as in LaserCortex/Cost.lean), and overlays discrete samples on continuous `eml` slices.

Core goals:
- Reproduce logexp activation slice(s) and mark local extrema.
- Enumerate binary trees up to `n=5` internal nodes and compute Loday coords.
- Build Tamari Hasse graph edges (single right rotations).
- Compute discrete Φ under the repo's `nodeParam` settings and color the Hasse graph.
- Overlay discrete tree samples (projected scalar) on the chosen `eml` slice.


In [ ]:
# Core imports
import math
import itertools
from collections import namedtuple
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

# Make plots render inline (for Jupyter)
%matplotlib inline


## Binary tree representation and enumerator
We use a simple tuple representation: a Leaf is `None`, and a Node is `(left, right)` where `left` and `right` are subtrees. The `size` is number of internal nodes.


In [ ]:
Leaf = None
def node(left, right):
    return (left, right)

def size(t):
    if t is None:
        return 0
    l,r = t
    return 1 + size(l) + size(r)

# enumerate all full binary trees with exactly n internal nodes
from functools import lru_cache
@lru_cache(None)
def trees_of_size(n):
    # returns list of trees with n internal nodes
    if n == 0:
        return [None]
    res = []
    # choose left subtree node count k from 0..n-1, right will be n-1-k
    for k in range(n):
        lefts = trees_of_size(k)
        rights = trees_of_size(n-1-k)
        for L in lefts:
            for R in rights:
                res.append(node(L,R))
    return res

# quick test
for n in range(6):
    print(n, len(trees_of_size(n)))


## Loday coordinates (as in LodayCoords.lean)
Each internal node contributes the number of leaves in its left subtree, visited in prefix order.


In [ ]:
def num_leaves(t):
    if t is None:
        return 1
    l,r = t
    return num_leaves(l) + num_leaves(r)

def loday_coord(t):
    if t is None:
        return []
    l,r = t
    return [num_leaves(l)] + loday_coord(l) + loday_coord(r)

# test and show for n=3 trees
trees = trees_of_size(3)
for t in trees:
    print(t, 'leaves=', num_leaves(t), 'loday=', loday_coord(t))


## Tamari single rotation successors (contracts_one_successors)
Implements the same successor generation as in EMLRegistry.lean.


In [ ]:
def contracts_one_successors(t):
    # returns list of trees reachable from t by a single contracts_one step
    res = []
    if t is None:
        return []
    l,r = t
    # case Node (Node a b) c -> Node a (Node b c)
    if l is not None:
        a,b = l
        # rotation at root when left is a Node
        res.append(node(a, node(b, r)))
    # rotations in left subtree
    if l is not None:
        for l2 in contracts_one_successors(l):
            res.append(node(l2, r))
    # rotations in right subtree
    if r is not None:
        for r2 in contracts_one_successors(r):
            res.append(node(l, r2))
    # remove duplicates by structural equality
    unique = []
    for x in res:
        if not any(structural_eq(x,y) for y in unique):
            unique.append(x)
    return unique

def structural_eq(a,b):
    # compare nested tuples structurally
    return a == b

# small test
t = node(node(node(None,None), None), None)  # ((Leaf Leaf) Leaf) Leaf
print('tree', t)
print('successors', contracts_one_successors(t))


## Discrete NodeCost and Φ (mirror of Cost.lean nodeParam)
We'll implement the same logic types and the NodeCost.apply function.


In [ ]:
# Define logic types as strings and nodeParam map matching Cost.lean
LogicNames = ['Classical','Fuzzy','ManyValued','Paraconsistent','Temporal','Deontic','Epistemic','Quantum','Intuitionistic','Relevance','Free','Infinitary','Modal','Spacetime','Boolean']
nodeParam = {
    'Classical':      {'leftWeight':1,'rightDiv':1,'bias':1},
    'Fuzzy':          {'leftWeight':1,'rightDiv':2,'bias':1},
    'ManyValued':     {'leftWeight':1,'rightDiv':1,'bias':1},
    'Paraconsistent': {'leftWeight':2,'rightDiv':1,'bias':1},
    'Temporal':       {'leftWeight':2,'rightDiv':1,'bias':1},
    'Deontic':        {'leftWeight':1,'rightDiv':2,'bias':1},
    'Epistemic':      {'leftWeight':1,'rightDiv':2,'bias':1},
    'Quantum':        {'leftWeight':1,'rightDiv':1,'bias':1},
    'Intuitionistic': {'leftWeight':1,'rightDiv':0,'bias':1},
    'Relevance':      {'leftWeight':1,'rightDiv':1,'bias':1},
    'Free':           {'leftWeight':1,'rightDiv':0,'bias':1},
    'Infinitary':     {'leftWeight':1,'rightDiv':1,'bias':1},
    'Modal':          {'leftWeight':1,'rightDiv':1,'bias':1},
    'Spacetime':      {'leftWeight':2,'rightDiv':1,'bias':1},
    'Boolean':        {'leftWeight':1,'rightDiv':0,'bias':1},
}

def node_cost_apply(c, a, b):
    # c: dict with leftWeight,rightDiv,bias; a,b ints
    return c['bias'] + c['leftWeight'] * a + (b // (c['rightDiv'] + 1))

from functools import lru_cache
@lru_cache(None)
def Phi(logic, t):
    c = nodeParam[logic]
    if t is None:
        return 0
    l,r = t
    return node_cost_apply(c, Phi(logic, l), Phi(logic, r))

# quick test
for logic in ['Classical','Paraconsistent','Fuzzy','Intuitionistic']:
    print(logic, [Phi(logic,t) for t in trees_of_size(3)])


## Build Tamari Hasse graph for a fixed size n and color by Φ
We'll construct a directed graph where edges are single rotations; for visualization we will draw an undirected version with node coloring.


In [ ]:
def build_tamari_graph(n, logic='Classical'):
    trees = trees_of_size(n)
    # canonical string repr for structural equality and mapping
    def repr_tree(t):
        if t is None:
            return '0'
        l,r = t
        return '1' + repr_tree(l) + repr_tree(r)
    id_map = {repr_tree(t):t for t in trees}
    G = nx.DiGraph()
    for t in trees:
        key = repr_tree(t)
        G.add_node(key, tree=t, phi=Phi(logic,t), loday=loday_coord(t))
    for t in trees:
        key = repr_tree(t)
        for s in contracts_one_successors(t):
            sk = repr_tree(s)
            if sk in id_map:
                G.add_edge(key, sk)
    return G

G = build_tamari_graph(4, logic='Paraconsistent')
print('nodes', len(G.nodes()), 'edges', len(G.edges()))

# visualize with matplotlib (simple layout)
plt.figure(figsize=(10,6))
H = nx.Graph(G)
pos = nx.spring_layout(H, seed=1)
phis = np.array([G.nodes[n]['phi'] for n in G.nodes()])
nx.draw_networkx_edges(H, pos, alpha=0.3)
nodes = nx.draw_networkx_nodes(H, pos, node_size=300, node_color=phis, cmap='viridis')
nx.draw_networkx_labels(H, pos, font_size=8)
plt.colorbar(nodes)


## Continuous `eml` slice reproduction and overlay
We will plot several candidate parametric slices `y(x)` and look for the peak→trough→rise shape. Then we project discrete trees to scalar `x` positions (e.g., via normalized first Loday coordinate) and overlay Phi points.


In [ ]:
def eml(x,y):
    return math.exp(x) - math.log(y)

# candidate slices
def y1(x):
    return math.exp(x/2) + 0.1
def y2(x):
    return 1.0 + 0.5*math.sin(0.8*x) + 0.2*math.exp(-0.2*x)
def y3(x):
    return math.exp(-0.8*x) + 0.05

xs = np.linspace(-6,6,2000)
candidates = {'y=exp(x/2)+0.1':y1, 'y=1+0.5*sin+exp(-0.2x)':y2, 'y=exp(-0.8x)+0.05':y3}
fig, axes = plt.subplots(3,1, figsize=(10,12))
for ax,(name,f) in zip(axes, candidates.items()):
    ys = np.array([f(x) for x in xs])
    vals = np.array([np.exp(x) - np.log(y) for x,y in zip(xs,ys)])
    ax.plot(xs, vals, lw=2, label=name)
    # mark local extrema
    dv = np.gradient(vals, xs)
    extrema_idx = np.where(np.sign(dv[:-1]) != np.sign(dv[1:]))[0] + 1
    ax.scatter(xs[extrema_idx], vals[extrema_idx], color='red')
    ax.set_title(name)
    ax.grid(True)
    ax.legend()
plt.tight_layout()

# Choose one candidate (the one that visually matches the repo PNG)
chosen = y1
vals = np.array([np.exp(x) - np.log(chosen(x)) for x in xs])
# project discrete trees to a scalar: take first Loday coordinate normalized
n = 4
trees = trees_of_size(n)
def scalar_projection(t):
    lc = loday_coord(t)
    if len(lc)==0: return 0.0
    # use first coordinate normalized by num_leaves
    return lc[0]/(num_leaves(t))
projs = np.array([scalar_projection(t) for t in trees])
# map projection to x-range [-6,6]
xs_t = -6 + projs*(12)
phis_t = np.array([Phi('Paraconsistent',t) for t in trees])
plt.figure(figsize=(10,4))
plt.plot(xs, vals, lw=2, label='chosen slice')
plt.scatter(xs_t, [eml(x, chosen(x)) for x in xs_t], c=phis_t, cmap='plasma', s=100, edgecolor='k')
for x,p in zip(xs_t, trees):
    plt.text(x, eml(x, chosen(x))+0.5, str(loday_coord(p)), fontsize=8, rotation=45)
plt.colorbar(label='Phi (Paraconsistent)')
plt.title('Selected eml slice with discrete tree samples overlaid')
plt.grid(True)


## Save/export options
You can export the plots as PNG or interactive HTML. If you want me to commit this notebook into the repo and add a small gnuplot script to reproduce the original PNG, I can do that next.
